In [1]:
import warnings
warnings.filterwarnings('ignore')
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

print("Libraries ready!")

Libraries ready!


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from rdkit import Chem
from rdkit.Chem import AllChem, Draw

import deepchem as dc
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, roc_curve
import shap
import joblib

print("All imports successful!")

In [8]:
# Load Tox21 via DeepChem
tasks, datasets, transformers = dc.molnet.load_tox21(
    featurizer='ECFP',  # Morgan fingerprints
    splitter='random'
)

train_dataset, valid_dataset, test_dataset = datasets

print(f"Tasks (endpoints): {tasks}")
print(f"Number of tasks: {len(tasks)}")
print(f"Train size: {len(train_dataset)}")
print(f"Test size: {len(test_dataset)}")

print(f"Train: {train_dataset.X.shape}")  # (n_samples, n_features)
print(f"Valid: {valid_dataset.X.shape}")
print(f"Test:  {test_dataset.X.shape}")

print(f"\nLabels shape: {train_dataset.y.shape}")  # (n_samples, 12)

print(f"\nFirst 5 SMILES:")
for smi in train_dataset.ids[:5]:
    print(f"  {smi}")

print(f"\nTasks: {tasks}")

import pandas as pd
df = pd.DataFrame(train_dataset.y[:10], columns=tasks)
print(f"\nFirst 10 rows of labels:")
print(df)

Tasks (endpoints): ['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53']
Number of tasks: 12
Train size: 6258
Test size: 783
Train: (6258, 1024)
Valid: (782, 1024)
Test:  (783, 1024)

Labels shape: (6258, 12)

First 5 SMILES:
  CC1(C)[C@@H](OC(=O)CCC(=O)[O-])CC[C@@]2(C)[C@H]1CC[C@]1(C)[C@@H]2C(=O)C=C2[C@@H]3C[C@@](C)(C(=O)[O-])CC[C@]3(C)CC[C@]21C
  Nc1cc(Cl)c(-c2cc(Cl)c(N)cc2Cl)cc1Cl
  C=CC(=O)OCCCCCC
  O=S1(=O)c2cccc3cccc(c23)N1CCCN1CCN(c2ccc(F)cc2)CC1
  CC(C)(C(=O)O)c1ccc(C(O)CCCN2CCC(C(O)(c3ccccc3)c3ccccc3)CC2)cc1

Tasks: ['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53']

First 10 rows of labels:
   NR-AR  NR-AR-LBD  NR-AhR  NR-Aromatase  NR-ER  NR-ER-LBD  NR-PPAR-gamma  \
0    0.0        0.0     0.0           0.0    0.0        0.0            0.0   
1    0.0        0.0     1.0           1.0    0.0        0.0  

In [ ]:
# Extract features and labels
X_train = train_dataset.X
y_train = train_dataset.y
w_train = train_dataset.w  # weights (0 = missing label for that compound/assay)

X_test = test_dataset.X
y_test = test_dataset.y
w_test = test_dataset.w

# Replace NaN labels with 0 (missing = treated as non-toxic, standard Tox21 convention)
y_train_clean = np.nan_to_num(y_train, nan=0)
y_test_clean = np.nan_to_num(y_test, nan=0)

print(f"Feature shape: {X_train.shape}")
print(f"Label shape:   {y_train.shape}")
print(f"\nClass balance per endpoint (train):")
for i, task in enumerate(tasks):
    pos = y_train_clean[:, i].sum()
    total = len(y_train_clean)
    print(f"  {task}: {pos:.0f} positive / {total} total ({100*pos/total:.1f}%)")

In [ ]:
# Train one XGBoost per endpoint with per-task scale_pos_weight
# scale_pos_weight = n_negative / n_positive corrects class imbalance
# Using a dict so app.py can load with models[task].predict_proba(fp)

models = {}
aucs = {}

print("Training per-task XGBoost models...")
for i, task in enumerate(tasks):
    y_tr = y_train_clean[:, i]
    y_te = y_test_clean[:, i]

    n_pos = (y_tr == 1).sum()
    n_neg = (y_tr == 0).sum()
    spw = n_neg / max(n_pos, 1)

    clf = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        scale_pos_weight=spw,
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1,
    )
    clf.fit(X_train, y_tr)
    models[task] = clf

    # Evaluate only on compounds that have a label for this endpoint
    mask = w_test[:, i] != 0
    y_pred = clf.predict_proba(X_test[mask])[:, 1]
    auc = roc_auc_score(y_te[mask], y_pred)
    aucs[task] = auc
    print(f"  {task:20s} AUC = {auc:.3f}  (spw={spw:.1f})")

joblib.dump(models, '../model/multitask_xgb.pkl')
print(f"\nAll models saved. Mean AUC: {np.mean(list(aucs.values())):.3f}")

## Evaluation — AUC per endpoint

In [ ]:
results = pd.DataFrame({
    'Endpoint': list(aucs.keys()),
    'Test AUC': [round(v, 3) for v in aucs.values()]
}).sort_values('Test AUC', ascending=False).reset_index(drop=True)

print(results.to_string(index=False))
print(f"\nMean AUC: {results['Test AUC'].mean():.3f}")
print(f"Min AUC:  {results['Test AUC'].min():.3f}  ({results.iloc[-1]['Endpoint']})")
print(f"Max AUC:  {results['Test AUC'].max():.3f}  ({results.iloc[0]['Endpoint']})")

## ROC Curves — all 12 endpoints

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 11))
axes = axes.flatten()

for i, task in enumerate(tasks):
    mask = w_test[:, i] != 0
    y_pred = models[task].predict_proba(X_test[mask])[:, 1]
    fpr, tpr, _ = roc_curve(y_test_clean[mask, i], y_pred)
    auc = aucs[task]

    axes[i].plot(fpr, tpr, lw=2, label=f'AUC = {auc:.3f}')
    axes[i].plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
    axes[i].set_title(task, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('FPR')
    axes[i].set_ylabel('TPR')
    axes[i].legend(loc='lower right', fontsize=9)
    axes[i].set_xlim([0, 1])
    axes[i].set_ylim([0, 1.02])

plt.suptitle(f'ROC Curves — 12 Tox21 Endpoints (Mean AUC = {np.mean(list(aucs.values())):.3f})',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../figures/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## SHAP — SR-ARE deep dive

SR-ARE (Stress Response – Antioxidant Response Element) measures activation of the Nrf2/ARE pathway, a key indicator of oxidative stress-induced toxicity and a clinically relevant hepatotoxicity signal.

In [ ]:
task_idx = tasks.index('SR-ARE')
clf_srare = models['SR-ARE']

# SHAP TreeExplainer is exact and fast for XGBoost
explainer = shap.TreeExplainer(clf_srare)
shap_values = explainer.shap_values(X_test)

# Summary plot: feature importance + direction
plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values, X_test,
    max_display=20,
    show=False,
    plot_type='dot'
)
plt.title('SHAP Summary — SR-ARE endpoint', fontsize=13)
plt.tight_layout()
plt.savefig('../figures/shap_summary_SR_ARE.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Bar plot: mean absolute SHAP values
plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_values, X_test,
    max_display=20,
    show=False,
    plot_type='bar'
)
plt.title('SHAP Feature Importance (mean |SHAP|) — SR-ARE', fontsize=13)
plt.tight_layout()
plt.savefig('../figures/shap_bar_SR_ARE.png', dpi=150, bbox_inches='tight')
plt.show()

## Validation on known molecules

Sanity check: do the predictions match what we know about these compounds?

In [ ]:
known_molecules = {
    "Paracetamol":    "CC(=O)Nc1ccc(O)cc1",
    "Aspirin":        "CC(=O)Oc1ccccc1C(=O)O",
    "Troglitazone":   "Cc1c(C)c2c(c(C)c1OCC1(C)SC(=O)NC1=O)CCC(C)(C)O2",
    "Caffeine":       "Cn1cnc2c1c(=O)n(C)c(=O)n2C",
    "Aflatoxin B1":   "O=c1oc2c(OC)cc3c(c2c2c1[C@@H]1C=C[C@@H](O1)O2)OCO3",
    "Ethanol":        "CCO",
}

def predict_all_endpoints(smiles, task_name='SR-ARE'):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024)
    x = np.array(fp).reshape(1, -1)
    return models[task_name].predict_proba(x)[0, 1]

print(f"{'Molecule':<18} {'SR-ARE score':>14} {'Prediction':>12}")
print("-" * 48)
for name, smi in known_molecules.items():
    score = predict_all_endpoints(smi, 'SR-ARE')
    pred = "HIGH RISK" if score > 0.5 else "LOW RISK"
    print(f"{name:<18} {score:>14.3f} {pred:>12}")